# Grokking
### *Generalization Beyond Overfitting on Small Algorithmic Datasets* — Power et al. 2022

**Paper:** [arXiv:2201.02177](https://arxiv.org/abs/2201.02177)

---

## The phenomenon

Train a small transformer to compute $(a + b) \bmod 97$.  Use 50% of all
$97^2 = 9{,}409$ pairs for training.

Watch what happens to validation accuracy over time:

```
Step    1,000 : train_acc = 100%,  val_acc =   1%   ← memorised!
Step   10,000 : train_acc = 100%,  val_acc =   2%   ← still memorised
Step   40,000 : train_acc = 100%,  val_acc =  99%   ← GROKKING!
```

The model first memorises the training set, then — **thousands of steps later** —
suddenly generalises. This is **grokking**.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

from mineo.experiments.grokking import GrokkingExperiment, ModularDataset
from mineo.visualization.plots import plot_grokking_curves, plot_fourier_spectrum

print(f"PyTorch {torch.__version__}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")


## The dataset: modular arithmetic

We train on the task $(a + b) \bmod p$ for prime $p = 97$.

- Vocabulary: integers $\{0, \ldots, 96\}$ plus `+` and `=` tokens (99 tokens total)
- Input sequence: `[a, +, b, =]`  (length 4)
- Target: predict $c = (a + b) \bmod 97$ at the `=` position

With 50% of pairs for training, the task is **hard enough to require generalisation**
but **small enough to run on a laptop**.


In [ ]:
# Explore the dataset
p = 97
ds = ModularDataset(p=p, train_frac=0.5, op='+')

print(f"Vocabulary size : {ds.vocab_size}")
print(f"Train pairs     : {len(ds.X_tr)}")
print(f"Val pairs       : {len(ds.X_val)}")
print()

# Show a few examples (tokens: 0..p-1 = numbers, p = '+', p+1 = '=')
OP_TOKEN = p
EQ_TOKEN = p + 1
print("Example input sequences → answer:")
for i in range(5):
    a, op, b, eq = ds.X_tr[i].tolist()
    c = ds.Y_tr[i].item()
    print(f"  [{a}, '+', {b}, '=']  →  {c}  (check: ({a}+{b}) mod {p} = {c})")


## Why does grokking happen?

### Two competing solutions

The model can solve the task in (at least) two ways:

1. **Memorisation** — store each $(a, b) \to c$ mapping as a lookup table.
   - Works perfectly on training data
   - Requires large, unstructured weights
   - Fails on unseen pairs

2. **Generalisation** — learn the underlying algorithm (modular addition).
   - Requires discovering the mathematical structure
   - Works on any $(a, b)$ pair
   - Implemented with smaller, more regular weights

### The role of weight decay

Weight decay adds an $L_2$ penalty: $\mathcal{L}_{total} = \mathcal{L}_{task} + \lambda \|\theta\|^2$.

This **continuously shrinks** the model's weights. Large-norm memorisation solutions
get penalised more than compact generalisation solutions. Over thousands of steps,
weight decay slowly prunes away the memorisation solution until the generalisation
solution emerges.

**Without weight decay, grokking does not happen** (the model stays memorised forever).


## Training

We train with strong weight decay ($\lambda = 1.0$), AdamW optimiser, and
log train/val accuracy every 1,000 steps.

> **Time estimate:** ~5–10 min on CPU for 50k steps.
> Use `n_steps=5_000` for a quick smoke test (model may not fully grok).


In [ ]:
N_STEPS = 50_000  # reduce to 5_000 for a quick test

exp = GrokkingExperiment(
    p            = 97,
    op           = '+',
    train_frac   = 0.5,
    d_model      = 128,
    n_layers     = 2,
    n_heads      = 4,
    weight_decay = 1.0,   # critical — set to 0 to disable grokking
    lr           = 1e-3,
    n_steps      = N_STEPS,
    batch_size   = 512,
    device       = device,
    verbose      = True,
)

print(f"Model parameters: {exp.model.n_params:,}")


In [ ]:
history = exp.train(log_interval=1_000)


In [ ]:
fig = plot_grokking_curves(history)
plt.show()


### What you should see

- **Left panel (loss):** Train loss drops rapidly. Val loss stays high, then collapses.
- **Right panel (accuracy):** Train accuracy hits 100% early.  Val accuracy stays near
  $1/97 \approx 1\%$ (random chance), then **jumps sharply to 100%** — grokking!

The green dashed line marks the grokking transition.

> If you used fewer steps, val accuracy may not have jumped yet.
> Run with `N_STEPS = 50_000` for the full effect.


## Mechanistic interpretation: Fourier features

*Nanda et al. (2023) — [arXiv:2301.05217](https://arxiv.org/abs/2301.05217)*

After grokking, the model uses a beautiful mathematical trick.  It represents
integer $k$ as a **point on a circle**:

$$\text{emb}(k) \approx \left[\cos\!\left(\frac{2\pi f k}{p}\right),\; \sin\!\left(\frac{2\pi f k}{p}\right), \ldots\right]$$

for a small set of dominant frequencies $f$.

Modular addition then becomes **angle addition** on the unit circle:

$$\cos\!\left(\frac{2\pi f (a+b)}{p}\right) = \cos\!\left(\frac{2\pi f a}{p}\right)\cos\!\left(\frac{2\pi f b}{p}\right) - \sin\!\left(\frac{2\pi f a}{p}\right)\sin\!\left(\frac{2\pi f b}{p}\right)$$

This is a **compact, structured representation** — hence small weights, hence favoured
by weight decay.

We can see this in the **Fourier power spectrum** of the learned embeddings.


In [ ]:
fourier = exp.fourier_analysis(n_freqs=15)
print(f"Top Fourier frequencies: {fourier['top_freqs']}")
print("(After grokking, a handful of frequencies dominate)")


In [ ]:
fig = plot_fourier_spectrum(
    fourier['power_spectrum'],
    top_freqs=fourier['top_freqs'][:5],
    title='Fourier spectrum of token embeddings\n(dominant peaks = Fourier representation of modular arithmetic)',
)
plt.show()


### What you should see

After grokking: a few sharp peaks in the spectrum (highlighted in red).
Before grokking: a flat/noisy spectrum — no structure.

This is direct evidence that the model has learned to represent integers as
points on a circle rather than memorised lookup tables.

---

## Ablation: weight decay matters

Run a quick comparison with `weight_decay=0`:


In [ ]:
# Quick ablation: train without weight decay
exp_no_wd = GrokkingExperiment(
    p=97, n_steps=10_000, weight_decay=0.0,
    device=device, verbose=False,
)
hist_no_wd = exp_no_wd.train(log_interval=1_000)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(hist_no_wd['steps'], hist_no_wd['val_acc'],
        label='val acc (weight_decay=0)', color='crimson', linestyle='--')
if history['val_acc']:
    ax.plot(history['steps'][:len(hist_no_wd['steps'])],
            history['val_acc'][:len(hist_no_wd['steps'])],
            label='val acc (weight_decay=1.0)', color='steelblue')
ax.set_xlabel('Step')
ax.set_ylabel('Val accuracy')
ax.set_title('Grokking requires weight decay')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

print(f"Final val acc (wd=0):   {hist_no_wd['val_acc'][-1]:.3f}")
if history['val_acc']:
    print(f"Final val acc (wd=1.0): {history['val_acc'][-1]:.3f}")


## Summary

| Concept | Key point |
|---|---|
| Grokking | Generalisation can happen thousands of steps *after* memorisation |
| Mechanism | Weight decay shrinks memorisation solutions; generalisation solutions survive |
| Representation | The model learns Fourier/circular features for modular arithmetic |
| Hyper-parameters | `weight_decay`, `train_frac`, and `n_steps` all affect the grokking delay |

### Further reading
- Power et al. 2022 — [arXiv:2201.02177](https://arxiv.org/abs/2201.02177) (original grokking paper)
- Nanda et al. 2023 — [arXiv:2301.05217](https://arxiv.org/abs/2301.05217) (mechanistic interpretation)
- Liu et al. 2022 — "Towards Understanding Grokking" — theoretical analysis
